# ReplayAxis frame comparison

Load a comparison downloaded from the evidence view and inspect how the two frames reached their decisions. Run this notebook next to the bundled synthetic sample, or set `export_path` to your own downloaded JSON file.

The notebook uses only the Python standard library. Frame numbers in the view are one-based. Source frame IDs preserve their original camera and zero-based frame index.

In [ ]:
import json
from pathlib import Path

export_path = Path("replayaxis-frames-4-5.json")
comparison = json.loads(export_path.read_text(encoding="utf-8"))
if comparison.get("schema_version") != 1:
    raise ValueError("This notebook expects a ReplayAxis comparison with schema_version 1.")

left, right = comparison["left"], comparison["right"]

## Reconstruction decisions

The score margin is the difference between the top two candidate groups' inlier counts when both groups exist. A null position means the estimator withheld the reconstruction.

In [ ]:
for frame in (left, right):
    result = frame["reconstruction"]
    print(f"Frame {frame['frame_number']} at {frame['timestamp_s'] * 1000:.2f} ms")
    margin = result["consensus"]["score_margin"]
    print(f"  Status: {result['status']}")
    print(f"  Score margin: {margin if margin is not None else 'Unavailable'}")
    print(f"  Position (m): {result['point_m'] if result['point_m'] is not None else 'Withheld'}")
    print(f"  {frame['explanation']}")

## Camera-by-camera evidence

“Undecided” means the observation exists but the estimator did not assign a per-camera decision. “No observation” means that camera has no observation in the selected frame.

In [ ]:
def observations_by_camera(frame):
    return {observation["camera_id"]: observation
            for observation in frame["reconstruction"]["observations"]}


def describe(observation):
    if observation is None:
        return "No observation"
    decision = observation.get("decision", "Undecided")
    return f"{decision.capitalize()} ({observation['frame_id']})"


left_views = observations_by_camera(left)
right_views = observations_by_camera(right)
print(f"{'Camera':<8} {'Frame ' + str(left['frame_number']):<25} Frame {right['frame_number']}")
for camera in sorted(left_views.keys() | right_views.keys()):
    print(f"{camera:<8} {describe(left_views.get(camera)):<25} {describe(right_views.get(camera))}")

With the bundled example, frame 4 withholds the position after a tie. Frame 5 recovers a position using cameras A, B, and C, and rejects camera D’s observation.